# 🐑 Sheep Activity Classifier
**Clasificación multiclase (5 clases) de actividades de ovejas · Kaggle Notebook**

### Estructura esperada
```
/kaggle/working/
├── preprocess.py
├── dataset.py
├── model.py
├── sheep_classifier.ipynb  ← este archivo

/kaggle/input/<dataset>/
├── train/          # videos .mov
├── test/           # videos .mov
└── train.csv
```

### Pipeline
```
Video → 480p → 16 frames → YOLO crop → ViT-B/16 → Temporal Attention → MLP → 5 clases
```

## 0 · Instalación de dependencias

In [2]:
# Kaggle ya tiene torch, torchvision, numpy, pandas, sklearn instalados
# Solo instalamos lo que falta
!pip install -q timm>=0.9.0 ultralytics>=8.0.0 albumentations einops

## 1 · Verificación de GPU y configuración global

In [21]:
import os, sys, math, random
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.amp import GradScaler, autocast   # Mixed precision
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm

# ── GPU check ────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "⚠️  GPU no disponible. Activa la GPU en Settings → Accelerator."

DEVICE = torch.device("cuda")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM total : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"   VRAM libre : {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
print(f"   PyTorch    : {torch.__version__}")
print(f"   CUDA       : {torch.version.cuda}")

✅ GPU: Tesla T4
   VRAM total : 15.6 GB
   VRAM libre : 15.0 GB
   PyTorch    : 2.10.0+cu128
   CUDA       : 12.8


In [15]:
# ── Rutas ─────────────────────────────────────────────────────────────
WORKING_DIR   = Path("/kaggle/working")
INPUT_DIR    = Path("/kaggle/input")   

# Data
TRAIN_VIDEO_DIR = WORKING_DIR / "train"
LABEL_CSV = INPUT_DIR / "datasets/jeffreyamc/sheep-labels/train.csv"

print(f"DATA_DIR: {TRAIN_VIDEO_DIR}")
print(f"LABEL_CSV: {LABEL_CSV}")

# Scripts           
MODULES_DIR = INPUT_DIR / "datasets/jeffreyamc/modules/"

print(f"MODULES_DIR {MODULES_DIR}")

PROCESSED_DIR = WORKING_DIR / "processed"
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Hiperparámetros ───────────────────────────────────────────────────
CFG = {
    # Preprocesamiento
    "n_frames"       : 16,
    "video_ext"      : ".mov",
    # Modelo
    "num_classes"    : 5,
    "dropout"        : 0.4,
    "unfreeze_blocks": 4,
    # Entrenamiento
    "epochs"         : 50,
    "batch_size"     : 16,       # Con GPU T4 16 GB → batch 16 es seguro
    "lr"             : 1e-4,
    "weight_decay"   : 0.05,
    "warmup_epochs"  : 5,
    "patience"       : 10,
    "val_fraction"   : 0.15,
    "mixup_prob"     : 0.5,
    "mixup_alpha"    : 0.4,
    "max_grad_norm"  : 1.0,
    "label_smoothing": 0.1,
    "num_workers"    : 2,        # Kaggle recomienda 2 workers
    "seed"           : 42,
    # Inferencia
    "use_tta"        : True,
    "tta_augments"   : 4,
}

# Reproducibilidad
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # True si el input size es fijo (más rápido)

set_seed(CFG["seed"])

# cudnn benchmark acelera cuando el tamaño de entrada es constante
# Lo activamos después del set_seed para inferencia
print("\nConfiguración:")
for k, v in CFG.items():
    print(f"  {k:<20} = {v}")

DATA_DIR: /kaggle/working/train
LABEL_CSV: /kaggle/input/datasets/jeffreyamc/sheep-labels/train.csv
MODULES_DIR /kaggle/input/datasets/jeffreyamc/modules

Configuración:
  n_frames             = 16
  video_ext            = .mov
  num_classes          = 5
  dropout              = 0.4
  unfreeze_blocks      = 4
  epochs               = 50
  batch_size           = 16
  lr                   = 0.0001
  weight_decay         = 0.05
  warmup_epochs        = 5
  patience             = 10
  val_fraction         = 0.15
  mixup_prob           = 0.5
  mixup_alpha          = 0.4
  max_grad_norm        = 1.0
  label_smoothing      = 0.1
  num_workers          = 2
  seed                 = 42
  use_tta              = True
  tta_augments         = 4


## 2 · Importar módulos del proyecto

In [9]:
# Los .py deben estar en /kaggle/working/ (súbelos como dataset o ejecútalos antes)
sys.path.insert(0, str(MODULES_DIR))

from preprocess import load_yolo, process_video
from dataset import (
    SheepActivityDataset,
    TemporalConsistentTransform,
    build_dataloaders,
    mixup_batch,
)
from model import (
    SheepActivityClassifier,
    LabelSmoothingCrossEntropy,
    build_model,
)

print("✅ Módulos importados correctamente")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Módulos importados correctamente


## 3 · Preprocesamiento de videos

In [10]:
# Carga YOLOv8 (se descarga automáticamente ~25 MB)
yolo = load_yolo("yolov8m.pt")
print("✅ YOLO listo")

[YOLO] Cargando modelo yolov8m.pt...
✅ YOLO listo


In [ ]:
def preprocess_split(
    video_dir: Path,
    split: str,
    yolo_model,
    n_frames: int = 16,
    video_ext: str = ".mov",
) -> None:
    """Procesa todos los videos de un split y guarda los frames."""
    out_root = PROCESSED_DIR / split
    video_files = sorted(video_dir.glob(f"*{video_ext}"))
    if not video_files:
        video_files = sorted(video_dir.glob("*.mp4"))

    print(f"\n[{split.upper()}] {len(video_files)} videos → {out_root}")
    failed = []

    for vf in tqdm(video_files, desc=f"Preprocesando {split}"):
        video_id = vf.stem
        out_dir  = out_root / video_id

        # Saltar si ya está procesado
        if out_dir.exists() and len(list(out_dir.glob("*.png"))) == n_frames:
            continue

        ok = process_video(
            video_path=str(vf),
            yolo_model=yolo_model,
            output_dir=str(out_dir),
            video_id=video_id,
            n_frames=n_frames,
        )
        if not ok:
            failed.append(vf.name)

    if failed:
        print(f"  ⚠️  {len(failed)} videos fallaron: {failed[:5]}")
    print(f"  ✅ Listo")


# Preprocesar train
preprocess_split(
    TRAIN_VIDEO_DIR, "train", yolo,
    CFG["n_frames"], CFG["video_ext"]
) 

# Preprocesar test
preprocess_split(
    TEST_VIDEO_DIR, "test", yolo,
    CFG["n_frames"], CFG["video_ext"]
)

# Liberar YOLO de memoria
del yolo
torch.cuda.empty_cache()
print("\n🧹 YOLO liberado de memoria")


[TRAIN] 100 videos → /kaggle/working/processed/train


Preprocesando train:   0%|          | 0/100 [00:00<?, ?it/s]

## 4 · DataLoaders

In [16]:
train_loader, val_loader = build_dataloaders(
    processed_dir = str(PROCESSED_DIR / "train"),
    label_csv     = str(LABEL_CSV),
    n_frames      = CFG["n_frames"],
    batch_size    = CFG["batch_size"],
    val_fraction  = CFG["val_fraction"],
    num_workers   = CFG["num_workers"],
    seed          = CFG["seed"],
)

# Verificar un batch
batch = next(iter(train_loader))
print(f"\nBatch shape : {batch['clip'].shape}")
print(f"Labels      : {batch['label'].tolist()}")
print(f"VRAM usada  : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

[Dataset] 85 samples cargados (is_train=True)
[Dataset] 15 samples cargados (is_train=False)
[DataLoader] Train: 85 | Val: 15

Batch shape : torch.Size([16, 16, 3, 224, 224])
Labels      : [2, 2, 1, 3, 4, 0, 3, 3, 2, 0, 3, 0, 1, 1, 2, 1]
VRAM usada  : 0.10 GB


## 5 · Modelo

In [17]:
model = build_model(
    num_classes          = CFG["num_classes"],
    n_frames             = CFG["n_frames"],
    dropout              = CFG["dropout"],
    unfreeze_last_n_blocks = CFG["unfreeze_blocks"],
    device               = "cuda",
)

# Compilar el modelo para Kaggle (PyTorch 2.x, reduce latencia ~20%)
if torch.__version__ >= "2.0.0":
    try:
        model = torch.compile(model, mode="reduce-overhead")
        print("✅ torch.compile activado (PyTorch 2.x)")
    except Exception as e:
        print(f"⚠️  torch.compile no disponible: {e}")

print(f"\nVRAM tras cargar modelo: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[Model] Parámetros totales: 85,798,656 | Entrenables: 28,353,024 (33.0%)
✅ torch.compile activado (PyTorch 2.x)

VRAM tras cargar modelo: 0.45 GB


## 6 · Entrenamiento

In [25]:
# ── Utilidades de entrenamiento ───────────────────────────────────────

def cosine_warmup_scheduler(optimizer, warmup_epochs: int, total_epochs: int):
    """Warmup lineal + cosine decay."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch + 1) / float(warmup_epochs)
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None

    def step(self, score: float) -> bool:
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, criterion, scaler) -> dict:
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="  Train", leave=False):
        clips  = batch["clip"].to(DEVICE, non_blocking=True)   # (B, N, C, H, W)
        labels = batch["label"].to(DEVICE, non_blocking=True)  # (B,)

        # Mixup
        use_mixup = (np.random.random() < CFG["mixup_prob"])
        if use_mixup:
            clips, labels_soft = mixup_batch(
                clips, labels, CFG["num_classes"], CFG["mixup_alpha"]
            )

        optimizer.zero_grad(set_to_none=True)  # Más eficiente que zero_grad()

        # Mixed precision (FP16) → duplica throughput en GPU Kaggle
        with autocast("cuda"):
            logits = model(clips)
            loss = criterion(logits, labels_soft if use_mixup else labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch["label"].numpy())

    return {
        "loss"    : total_loss / len(loader),
        "macro_f1": f1_score(all_labels, all_preds, average="macro", zero_division=0),
    }


@torch.no_grad()
def validate(model, loader, criterion) -> dict:
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for batch in tqdm(loader, desc="  Val  ", leave=False):
        clips  = batch["clip"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with autocast("cuda"):
            logits = model(clips)
            loss   = criterion(logits, labels)

        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    f1_per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
    return {
        "loss"        : total_loss / len(loader),
        "macro_f1"    : float(f1_per_class.mean()),
        "f1_per_class": f1_per_class.tolist(),
    }

print("✅ Funciones de entrenamiento definidas")

✅ Funciones de entrenamiento definidas


In [26]:
# ── Criterio ──────────────────────────────────────────────────────────
criterion = LabelSmoothingCrossEntropy(
    smoothing=CFG["label_smoothing"],
    num_classes=CFG["num_classes"],
)

# ── Optimizador con weight decay diferenciado ─────────────────────────
# No aplicar WD a bias y LayerNorm (práctica estándar ViT)
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if "bias" in name or "norm" in name:
        no_decay.append(param)
    else:
        decay.append(param)

optimizer = AdamW(
    [{"params": decay,    "weight_decay": CFG["weight_decay"]},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=CFG["lr"],
    betas=(0.9, 0.999)
)

# ── Scheduler ─────────────────────────────────────────────────────────
scheduler = cosine_warmup_scheduler(
    optimizer,
    warmup_epochs=CFG["warmup_epochs"],
    total_epochs=CFG["epochs"],
)

# ── GradScaler para mixed precision ───────────────────────────────────
scaler = GradScaler("cuda")

# ── Early stopping ────────────────────────────────────────────────────
early_stop = EarlyStopping(patience=CFG["patience"])

print("✅ Optimizador, scheduler y scaler listos")

✅ Optimizador, scheduler y scaler listos


In [27]:
# ── Loop principal ────────────────────────────────────────────────────
BEST_PATH = CHECKPOINT_DIR / "best_model.pt"
history   = []
best_f1   = 0.0

print(f"{'='*65}")
print(f"  Entrenamiento: {CFG['epochs']} épocas | LR={CFG['lr']} | BS={CFG['batch_size']}")
print(f"  GPU: {torch.cuda.get_device_name(0)}  |  Mixed Precision FP16: ON")
print(f"{'='*65}\n")

for epoch in range(1, CFG["epochs"] + 1):
    current_lr = optimizer.param_groups[0]["lr"]

    train_m = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    val_m   = validate(model, val_loader, criterion)
    scheduler.step()

    row = {
        "epoch"    : epoch,
        "lr"       : current_lr,
        "train_loss": train_m["loss"],
        "train_f1" : train_m["macro_f1"],
        "val_loss" : val_m["loss"],
        "val_f1"   : val_m["macro_f1"],
    }
    history.append(row)

    improved = "  ✓ BEST" if val_m["macro_f1"] > best_f1 else ""
    print(
        f"Epoch {epoch:3d}/{CFG['epochs']} | "
        f"LR={current_lr:.2e} | "
        f"Train L={train_m['loss']:.4f} F1={train_m['macro_f1']:.4f} | "
        f"Val L={val_m['loss']:.4f} F1={val_m['macro_f1']:.4f}{improved}"
    )
    print(f"  F1/clase: {[f'{v:.3f}' for v in val_m['f1_per_class']]}")

    if val_m["macro_f1"] > best_f1:
        best_f1 = val_m["macro_f1"]
        torch.save({
            "epoch"       : epoch,
            "model_state" : model.state_dict(),
            "val_f1"      : best_f1,
            "cfg"         : CFG,
        }, BEST_PATH)

    if early_stop.step(val_m["macro_f1"]):
        print(f"\n⏹  Early stopping en época {epoch} (paciencia={CFG['patience']})")
        break

# Guardar histórico
hist_df = pd.DataFrame(history)
hist_df.to_csv(CHECKPOINT_DIR / "history.csv", index=False)
print(f"\n🏆 Mejor Val Macro F1: {best_f1:.4f}")
print(f"💾 Checkpoint: {BEST_PATH}")

  Entrenamiento: 50 épocas | LR=0.0001 | BS=16
  GPU: Tesla T4  |  Mixed Precision FP16: ON



  Train:   0%|          | 0/5 [00:00<?, ?it/s]

W0424 05:53:22.082000 55 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


  Val  :   0%|          | 0/1 [00:00<?, ?it/s]

Epoch   1/50 | LR=2.00e-05 | Train L=2.4896 F1=0.1639 | Val L=1.7894 F1=0.0944  ✓ BEST
  F1/clase: ['0.250', '0.000', '0.000', '0.222', '0.000']


...

  Val  :   0%|          | 0/1 [00:00<?, ?it/s]

Epoch  14/50 | LR=9.24e-05 | Train L=0.6420 F1=0.8532 | Val L=0.6107 F1=1.0000
  F1/clase: ['1.000', '1.000', '1.000', '1.000', '1.000']


  Train:   0%|          | 0/5 [00:00<?, ?it/s]

  Val  :   0%|          | 0/1 [00:00<?, ?it/s]

Epoch  15/50 | LR=9.05e-05 | Train L=0.7362 F1=0.9435 | Val L=0.5311 F1=1.0000
  F1/clase: ['1.000', '1.000', '1.000', '1.000', '1.000']

⏹  Early stopping en época 15 (paciencia=10)

🏆 Mejor Val Macro F1: 1.0000
💾 Checkpoint: /kaggle/working/checkpoints/best_model.pt


## 7 · Curvas de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss", color="steelblue")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"],   label="Val Loss",   color="tomato")
axes[0].set_title("Loss"); axes[0].set_xlabel("Época")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Macro F1
axes[1].plot(hist_df["epoch"], hist_df["train_f1"], label="Train F1", color="steelblue")
axes[1].plot(hist_df["epoch"], hist_df["val_f1"],   label="Val F1",   color="tomato")
axes[1].axhline(best_f1, color="green", linestyle="--", label=f"Best Val F1={best_f1:.4f}")
axes[1].set_title("Macro F1-Score"); axes[1].set_xlabel("Época")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(CHECKPOINT_DIR / "training_curves.png", dpi=120)
plt.show()

## 8 · Inferencia y generación del submission

In [ ]:
# ── Cargar el mejor checkpoint ────────────────────────────────────────
ckpt = torch.load(BEST_PATH, map_location=DEVICE)

inference_model = SheepActivityClassifier(
    num_classes            = CFG["num_classes"],
    n_frames               = CFG["n_frames"],
    dropout                = 0.0,   # Sin dropout en inferencia
    unfreeze_last_n_blocks = CFG["unfreeze_blocks"],
    use_temporal_attention = True,
).to(DEVICE)

# Cargar pesos (compatible con torch.compile)
state = ckpt["model_state"]
# Si el modelo fue compilado, los keys tienen prefijo '_orig_mod.' → limpiar
state = {k.replace("_orig_mod.", ""): v for k, v in state.items()}
inference_model.load_state_dict(state, strict=False)
inference_model.eval()

torch.backends.cudnn.benchmark = True  # Inferencia más rápida con tamaño fijo
print(f"✅ Checkpoint cargado · epoch={ckpt['epoch']} · val_f1={ckpt['val_f1']:.4f}")

In [ ]:
# ── Dataset de test ───────────────────────────────────────────────────
test_ds = SheepActivityDataset(
    processed_dir = str(PROCESSED_DIR / "test"),
    labels_df     = None,
    n_frames      = CFG["n_frames"],
    is_train      = False,
)
print(f"Test videos: {len(test_ds)}")

In [ ]:
# ── Inferencia con TTA ────────────────────────────────────────────────

def apply_tta(model, frames_pil: list, n_augments: int = 4) -> np.ndarray:
    """TTA: 1 pase sin aug + (n_augments-1) pases con aug → promedia softmax."""
    all_probs = []

    # Pase sin augmentation
    tf_val = TemporalConsistentTransform(is_train=False)
    clip = tf_val(frames_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad(), autocast("cuda"):
        all_probs.append(F.softmax(model(clip), dim=-1).cpu().numpy())

    # Pases con augmentation suave
    tf_aug = TemporalConsistentTransform(is_train=True)
    for _ in range(n_augments - 1):
        clip = tf_aug(frames_pil).unsqueeze(0).to(DEVICE)
        with torch.no_grad(), autocast("cuda"):
            all_probs.append(F.softmax(model(clip), dim=-1).cpu().numpy())

    return np.mean(all_probs, axis=0)  # (1, num_classes)


# ── Inferencia en batch estándar (sin TTA, más rápida) ────────────────
@torch.no_grad()
def run_batch_inference(model, dataset) -> pd.DataFrame:
    loader = DataLoader(
        dataset, batch_size=CFG["batch_size"],
        shuffle=False, num_workers=CFG["num_workers"], pin_memory=True,
    )
    ids, preds = [], []
    for batch in tqdm(loader, desc="Inferencia batch"):
        clips = batch["clip"].to(DEVICE, non_blocking=True)
        with autocast("cuda"):
            logits = model(clips)
        preds.extend(logits.argmax(1).cpu().tolist())
        ids.extend(batch["video_id"])
    return pd.DataFrame({"Id": ids, "Predicted": preds})


# ── Elegir modo ───────────────────────────────────────────────────────
if CFG["use_tta"]:
    print(f"Modo: TTA ({CFG['tta_augments']} augmentaciones)")
    results = []
    for idx in tqdm(range(len(test_ds)), desc="Inferencia TTA"):
        video_id = test_ds.samples[idx][0]
        frames   = test_ds._load_frames(video_id)
        probs    = apply_tta(inference_model, frames, CFG["tta_augments"])
        pred     = int(probs.argmax(axis=-1)[0])
        results.append({"Id": video_id, "Predicted": pred})
    submission = pd.DataFrame(results)
else:
    print("Modo: batch estándar")
    submission = run_batch_inference(inference_model, test_ds)

print(f"\n✅ {len(submission)} predicciones generadas")
print("Distribución de clases predichas:")
print(submission["Predicted"].value_counts().sort_index())

In [ ]:
# ── Guardar submission ────────────────────────────────────────────────
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"💾 Submission guardado en: {SUBMISSION_PATH}")
print("\nPrimeras filas:")
submission.head(10)

## 9 · Validación del formato del submission
Verifica que el CSV tenga el formato correcto antes de subirlo.

In [ ]:
df_check = pd.read_csv(SUBMISSION_PATH)

assert list(df_check.columns) == ["Id", "Predicted"], "❌ Columnas incorrectas"
assert df_check["Predicted"].between(0, 4).all(), "❌ Hay predicciones fuera del rango 0-4"
assert df_check["Id"].nunique() == len(df_check), "❌ Hay IDs duplicados"
assert df_check.isnull().sum().sum() == 0, "❌ Hay valores nulos"

print(f"✅ Submission válido")
print(f"   Filas        : {len(df_check)}")
print(f"   Clases únicas: {sorted(df_check['Predicted'].unique())}")
print(f"   IDs únicos   : {df_check['Id'].nunique()}")